# Investment Periods: Pathways, Lifetimes and Discounting

All models of these notebooks describe a **pathway**: the system develops over three investment periods, 2020, 2030 and 2040, and FINE optimizes all of them together. Since FINE knows all periods in advance, this is also called *perfect foresight*. This notebook explains how FINE handles time in such a model:

| Topic | Parameters |
|---|---|
| Investment periods | `startYear`, `numberOfInvestmentPeriods`, `investmentPeriodInterval` |
| How the total cost is computed | `interestRate`, `economicLifetime`, `NPVcontribution` |
| How long capacity exists | `technicalLifetime`, `floorTechnicalLifetime` |
| The end of the pathway | `annuityPerpetuity` |

## 1. Investment periods

Three parameters of the `EnergySystemModel` define the investment periods: `startYear` (2020), `numberOfInvestmentPeriods` (3) and `investmentPeriodInterval` (10 years). This gives the periods 2020, 2030 and 2040. Each period

- is a point in time at which new capacity can be built,
- is represented by **one year of operation**, with its own time series and costs where they are given per period,
- stands for the **10 years** that start with it: the period 2040 for the years 2040 to 2049.

Parameters that change over time, such as `investPerCapacity={2020: 1300, 2030: 1150, 2040: 1050}`, are given as dictionaries with exactly these years as keys. If a component has existing capacity (`stockCommissioning`), its cost dictionaries also need the years of the existing capacity.

In [1]:
import warnings

import pandas as pd

from tutorialHelpers import buildBasicTutorialModel

warnings.filterwarnings("ignore")  # keeps the output of this tutorial readable

In [2]:
esM = buildBasicTutorialModel()
esM.aggregateTemporally(n_clusters=8, segments=None)
esM.optimize(timeSeriesAggregation=True)

Set parameter OutputFlag to value 1


Set parameter Threads to value 3


Set parameter LogFile to value ""


Set parameter QCPDual to value 1


Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26300.2))


CPU model: Intel(R) Core(TM) Ultra 7 255U, instruction set [SSE2|AVX|AVX2]


... (71 outputs left out)

Solved with barrier


    3402    5.9738972e+05   0.000000e+00   0.000000e+00      1s


Solved in 3402 iterations and 0.59 seconds (0.88 work units)


Optimal objective  5.973897241e+05


## 2. How the total cost is computed

FINE minimizes the **net present value** of all costs: the costs of every year from 2020 to 2049, discounted to the start year 2020.

1. **Investments** are turned into yearly payments, the *annuities*: an investment is paid back over its `economicLifetime` with the `interestRate` of the component, like a loan.
2. For every period, FINE computes the **total annual cost** (`TAC` in the optimization summary): annuities, fixed and variable operation costs, and commodity costs. It applies to each of the 10 years of the period.
3. Each year *t* years after 2020 is **discounted** with the factor 1/(1+r)<sup>t</sup>, again with the interest rate *r* of the component. The year 2020 itself is not discounted.

The sum of the discount factors of the 10 years of a period gives the weight of its yearly costs:

In [3]:
interestRate = 0.08
discountWeights = pd.Series(
    {
        year: sum(
            1 / (1 + interestRate) ** t
            for t in range(year - esM.startYear, year - esM.startYear + 10)
        )
        for year in esM.investmentPeriodNames
    },
    name="Weight of the yearly costs",
)
discountWeights.round(3)

2020    7.247
2030    3.357
2040    1.555
Name: Weight of the yearly costs, dtype: float64

A cost of 1 million Euro per year during 2020 to 2029 counts as 7.2 million Euro in the total cost; the same cost during 2040 to 2049 only as 1.6 million Euro. Later costs count less.

The summary contains both values: `TAC` per year and `NPVcontribution`, the discounted contribution of the period to the total cost. Their ratio is exactly the weight above, e.g. for the PV in the north:

In [4]:
ratio = {}
for year in esM.investmentPeriodNames:
    summary = esM.getOptimizationSummary("SourceSinkModel", outputLevel=1, ip=year)
    pv = summary.loc["PV"].droplevel("Unit")["north"]
    ratio[year] = pv["NPVcontribution"] / pv["TAC"]
pd.Series(ratio, name="NPVcontribution / TAC of the PV (north)").round(3)

2020    7.247
2030    3.357
2040    1.555
Name: NPVcontribution / TAC of the PV (north), dtype: float64

All `NPVcontribution`s together give the objective value, as shown in the [Results and Plots notebook](33_results_and_plots.ipynb).

Two consequences of this way of computing:

- Since every component uses its own `interestRate` for the annuities **and** for discounting, components with different interest rates are discounted differently. In our model, all components use 8 %.
- Costs after the end of the pathway (2049) are **not counted**. Wind turbines built in 2040 pay only 10 of their 20 annuities within the pathway. This makes investments in the last period look cheaper than they are; section 4 shows how to change this.

## 3. Lifetimes and whole periods

Two lifetimes describe a component:

- **`economicLifetime`**: the number of years over which the investment is paid back (the annuities).
- **`technicalLifetime`**: how long the capacity exists. If not given, it is the same as the economic lifetime. It cannot be shorter than the economic lifetime.

Capacity can only be built and taken out of operation at the start of an investment period, so FINE rounds the technical lifetime to whole periods. By default (`floorTechnicalLifetime=True`), it rounds **down**. The PV has a lifetime of 25 years; with periods of 10 years, this becomes 20 years. PV built in 2020 is therefore available in 2020 and 2030 and is decommissioned in 2040, five years early:

In [5]:
pvSouth = pd.DataFrame(
    {
        year: esM.getOptimizationSummary("SourceSinkModel", outputLevel=1, ip=year)
        .loc["PV"]
        .droplevel("Unit")["south"][["capacity", "commissioning", "decommissioning"]]
        for year in esM.investmentPeriodNames
    }
)
pvSouth.columns.name = "PV in the south [GW]"
pvSouth.astype(float).round(1)

PV in the south [GW],2020,2030,2040
Property,,,
capacity,78.0,78.0,78.0
commissioning,78.0,0.0,78.0
decommissioning,0.0,0.0,78.0


The 78 GW of PV in the south are built in 2020, decommissioned in 2040 and built again in the same period. Rounded down, the PV also pays only 20 of its 25 annuities. With `floorTechnicalLifetime=False`, FINE rounds **up** instead: the PV lives 30 years and pays its annuity for 30 years, 5 more than the 25 years of its economic lifetime. Which way is better depends on the component; for the PV, both are 5 years away from the real lifetime. We change the PV and optimize again:

In [6]:
costRoundedDown = esM.objectiveValue

esM.updateComponent("PV", {"floorTechnicalLifetime": False})
esM.aggregateTemporally(n_clusters=8, segments=None)
esM.optimize(timeSeriesAggregation=True)

Set parameter OutputFlag to value 1


Set parameter Threads to value 3


Set parameter LogFile to value ""


Set parameter QCPDual to value 1


Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26300.2))


CPU model: Intel(R) Core(TM) Ultra 7 255U, instruction set [SSE2|AVX|AVX2]


... (74 outputs left out)

Solved with barrier


    2778    6.0172479e+05   0.000000e+00   0.000000e+00      1s


Solved in 2778 iterations and 0.62 seconds (0.86 work units)


Optimal objective  6.017247898e+05


In [7]:
pvSouthRoundedUp = pd.DataFrame(
    {
        year: esM.getOptimizationSummary("SourceSinkModel", outputLevel=1, ip=year)
        .loc["PV"]
        .droplevel("Unit")["south"][["capacity", "commissioning", "decommissioning"]]
        for year in esM.investmentPeriodNames
    }
)
pvSouthRoundedUp.columns.name = "PV in the south, rounded up [GW]"
pvSouthRoundedUp.astype(float).round(1)

"PV in the south, rounded up [GW]",2020,2030,2040
Property,,,
capacity,78.0,78.0,78.0
commissioning,78.0,0.0,0.0
decommissioning,0.0,0.0,0.0


In [8]:
pd.Series(
    {"Rounded down": costRoundedDown / 1000, "Rounded up": esM.objectiveValue / 1000},
    name="Total cost [billion Euro]",
).round(1)

Rounded down    597.4
Rounded up      601.7
Name: Total cost [billion Euro], dtype: float64

Rounded up, the PV built in 2020 is still available in 2040, so nothing has to be built again in 2040. The PV now pays its annuity for 30 instead of 20 years, and the total cost is about 4 billion Euro higher. Rounding is only an issue when lifetimes are not multiples of the period length; shorter periods (e.g. 5 years) reduce the error, but make the model larger.

## 4. The end of the pathway

As described in section 2, the total cost only contains the years up to 2049. With **`annuityPerpetuity=True`**, a parameter of the `EnergySystemModel`, FINE instead assumes that the yearly costs of the last period continue forever. The costs of the investments of the last period are then no longer cut off after ten years, and the system of 2040 has to be a good solution for the long run, not only for ten years. It requires an `interestRate` above 0 for all components. Note that the `NPVcontribution`s in the summaries do not contain this extension, so they no longer add up to the objective value.

## Things to try

- Optimize the model with `annuityPerpetuity=True` (a parameter of `buildBasicTutorialModel`). How do the capacities of 2040 change?
- Set the `interestRate` of the wind turbines to 0.04. They become cheaper, but their costs are also discounted less. Which effect is stronger?
- Give the gas power plants a `technicalLifetime` of 40 years (with their economic lifetime of 30 years). How long do the plants built in 2020 exist now?

The next notebook, [Binary Decisions](43_binary_decisions.ipynb), introduces decisions that can only be yes or no.